In [5]:
# Cell 1: 필수 라이브러리 설치
# Playwright가 설치되어 있지 않다면 아래 셀을 먼저 실행하세요.
%pip install playwright
!playwright install chromium


Note: you may need to restart the kernel to use updated packages.


In [9]:
# Cell 2: 라이브러리 import 및 설정
import json
import time
import re
import sys
import os
import subprocess
import tempfile
import inspect

# 노트북에서 시각 디버깅을 원하면 False로 설정하세요
NOTEBOOK_HEADLESS = True

# 수집할 검색어 개수
MAX_TRENDS = 80

# UI 텍스트 필터링 (제외할 텍스트 목록)
EXCLUDED_TEXTS = {
    "Trends", "트렌드 상태", "트렌드 분석", "검색", "탐색", "실시간 인기", "홈",
    "전 세계", "지금", "에서 무엇을 검색하고 있는지 알아보세요",
    "검색 관심도", "지난 24시간", "이(가) 인기 있는 이유는 무엇일까요?",
    "상세 데이터 검토", "트렌드 데이터팀", "선별한 문제와 이벤트",
    "트렌드 활용법", "언론사", "자선단체", "전 세계에서", "Google 트렌드를 어떻게 사용하고 있는지",
    "확인해보세요", "Google 트렌드란 무엇인가요?", "Google 트렌드의 기본사항",
    "데이터에 관해 알아보기", "로그인", "개인정보처리방침", "고급 Google 트렌드", "도움말", "의견 보내기"
}

# 구글 트렌드 URL
TREND_URL = "https://trends.google.co.kr/trending?geo=KR"

print(f"🔍 구글 트렌드 실시간 인기 검색어 크롤링")
print(f"🔗 URL: {TREND_URL}")
print(f"📦 최대 수집 개수: {MAX_TRENDS}개\n")


🔍 구글 트렌드 실시간 인기 검색어 크롤링
🔗 URL: https://trends.google.co.kr/trending?geo=KR
📦 최대 수집 개수: 80개



In [10]:
# Cell 3: 크롤링 함수 정의

def crawl_google_trends(headless=True, max_trends=80, excluded_texts=None):
    """
    구글 트렌드 실시간 인기 검색어를 크롤링합니다.
    
    Args:
        headless: 헤드리스 모드 여부
        max_trends: 최대 수집할 검색어 개수
        excluded_texts: 제외할 텍스트 집합 (None이면 EXCLUDED_TEXTS 사용)
        
    Returns:
        dict: {"total_trends": int, "trends": list}
    """
    if excluded_texts is None:
        excluded_texts = EXCLUDED_TEXTS
    
    trends = []
    found_keywords = set()
    
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=headless)
        page = browser.new_page()
        
        # 구글 트렌드 실시간 인기 페이지 접속
        url = "https://trends.google.co.kr/trending?geo=KR"
        print(f"접속 중: {url}")
        
        try:
            page.goto(url, wait_until="networkidle", timeout=60000)
            time.sleep(8)  # JavaScript 실행 대기
            
            # 트렌드 콘텐츠가 로드될 때까지 대기
            print("트렌드 콘텐츠 로드 대기 중...")
            try:
                # c-wiz 컴포넌트나 트렌드 항목이 나타날 때까지 대기
                page.wait_for_selector("c-wiz, [jsname], [jscontroller]", timeout=20000)
            except:
                pass
            
            # 추가 대기 (동적 콘텐츠 로드)
            time.sleep(5)
            
            # 스크롤하여 더 많은 트렌드 로드
            print("스크롤하여 더 많은 트렌드 로드 중...")
            for scroll_idx in range(20):
                page.evaluate("window.scrollBy(0, window.innerHeight)")
                time.sleep(2)
                # 스크롤 후 추가 대기
                if scroll_idx % 5 == 0:
                    time.sleep(3)
            
            page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            time.sleep(3)
            
            # 방법 0: 페이지 네비게이션 기반 테이블에서 키워드 추출
            print("\n0. 페이지 네비게이션 테이블에서 키워드 추출 중...")
            page.wait_for_timeout(1000)
            paged_round = 0
            while len(trends) < max_trends:
                paged_round += 1
                if paged_round > 20:
                    break
                rows = page.query_selector_all("tbody tr")
                new_items = 0
                for row in rows:
                    keyword_elem = row.query_selector(".mZ3RIc")
                    if not keyword_elem:
                        continue
                    text = keyword_elem.inner_text().strip()
                    if text and 1 < len(text) < 100:
                        if (text not in excluded_texts and
                            text not in found_keywords and
                            not any(excluded in text for excluded in excluded_texts)):
                            link_url = ""
                            link_elem = row.query_selector("a")
                            if link_elem:
                                href = link_elem.get_attribute("href") or ""
                                if href.startswith("http"):
                                    link_url = href
                                elif href.startswith("/"):
                                    link_url = f"https://trends.google.co.kr{href}"
                                elif href:
                                    link_url = f"https://trends.google.co.kr/{href}"
                            found_keywords.add(text)
                            trends.append({
                                "keyword": text,
                                "link": link_url
                            })
                            new_items += 1
                            if len(trends) >= max_trends:
                                break
                if len(trends) >= max_trends:
                    break
                if new_items == 0:
                    break
                selectors = [
                    "[aria-label*='다음 페이지']",
                    "[aria-label*='다음']",
                    "[aria-label*='Next page']",
                    "[aria-label*='Next']"
                ]
                next_button = None
                for selector in selectors:
                    try:
                        next_button = page.query_selector(selector)
                    except:
                        next_button = None
                    if next_button:
                        break
                if not next_button:
                    break
                try:
                    next_button.click()
                except:
                    break
                page.wait_for_timeout(2000)
            
            # 방법 1: .mZ3RIc 클래스에서 키워드 추출 (가장 정확한 방법)
            print("\n1. .mZ3RIc 클래스에서 키워드 추출 중...")
            trend_elements = page.query_selector_all(".mZ3RIc")
            
            for elem in trend_elements:
                try:
                    text = elem.inner_text().strip()
                    
                    if text and len(text) > 1 and len(text) < 100:
                        # UI 텍스트 필터링
                        if (text not in excluded_texts and
                            not text.startswith("http") and
                            text not in found_keywords and
                            not any(excluded in text for excluded in excluded_texts)):
                            
                            # 부모 링크 찾기
                            link_url = ""
                            try:
                                # 부모 요소 중 링크 찾기
                                parent = elem.evaluate_handle("el => el.closest('a')")
                                if parent:
                                    href = parent.get_attribute("href") or ""
                                    if href:
                                        if href.startswith("http"):
                                            link_url = href
                                        elif href.startswith("/"):
                                            link_url = f"https://trends.google.co.kr{href}"
                                        else:
                                            link_url = f"https://trends.google.co.kr/{href}"
                            except:
                                pass
                            
                            found_keywords.add(text)
                            trends.append({
                                "keyword": text,
                                "link": link_url
                            })
                            
                            if len(trends) >= max_trends:
                                break
                except:
                    continue
            
            # 방법 2: 모든 링크에서 키워드 추출 (백업 방법)
            if len(trends) < max_trends:
                print(f"\n2. 모든 링크에서 추가 키워드 추출 중... (현재 {len(trends)}개)")
                all_links = page.query_selector_all("a")
            
                # 링크에서 추출
                for link in all_links:
                    try:
                        href = link.get_attribute("href") or ""
                        text = link.inner_text().strip()
                        
                        # 모든 링크에서 텍스트 추출 (트렌드 관련 여부와 관계없이)
                        if text and len(text) > 2 and len(text) < 100:
                            # UI 텍스트 필터링
                            if (text not in excluded_texts and
                                not text.startswith("http") and
                                text not in found_keywords and
                                not any(excluded in text for excluded in excluded_texts)):
                                
                                # 링크 정규화
                                if href.startswith("http"):
                                    link_url = href
                                elif href.startswith("/"):
                                    link_url = f"https://trends.google.co.kr{href}"
                                elif href:
                                    link_url = f"https://trends.google.co.kr/{href}"
                                else:
                                    link_url = ""
                                
                                found_keywords.add(text)
                                trends.append({
                                    "keyword": text,
                                    "link": link_url
                                })
                                
                                if len(trends) >= max_trends:
                                    break
                    except:
                        continue
            
            # 방법 3: div, span 등 모든 요소에서 텍스트 추출 (최후 백업)
            if len(trends) < max_trends:
                print(f"\n3. 모든 요소에서 추가 텍스트 추출 중... (현재 {len(trends)}개)")
                
                # 모든 div, span, p 요소에서 텍스트 추출
                all_elements = page.query_selector_all("div, span, p, h1, h2, h3, h4, h5, h6")
                
                for elem in all_elements:
                    try:
                        text = elem.inner_text().strip()
                        
                        # 텍스트가 있고 적절한 길이인지 확인
                        if text and len(text) > 2 and len(text) < 100:
                            # 여러 줄인 경우 첫 번째 줄만
                            first_line = text.split("\n")[0].strip()
                            
                            # UI 텍스트 필터링
                            if (first_line not in excluded_texts and
                                first_line not in found_keywords and
                                not first_line.startswith("http") and
                                not any(excluded in first_line for excluded in excluded_texts)):
                                
                                # 링크 찾기
                                link_elem = elem.query_selector("a")
                                link_url = ""
                                if link_elem:
                                    href = link_elem.get_attribute("href") or ""
                                    if href:
                                        if href.startswith("http"):
                                            link_url = href
                                        elif href.startswith("/"):
                                            link_url = f"https://trends.google.co.kr{href}"
                                        else:
                                            link_url = f"https://trends.google.co.kr/{href}"
                                
                                found_keywords.add(first_line)
                                trends.append({
                                    "keyword": first_line,
                                    "link": link_url
                                })
                                
                                if len(trends) >= max_trends:
                                    break
                    except:
                        continue
            
            # 최종 결과 정리
            trends = trends[:max_trends]
            
            print(f"\n✅ 총 {len(trends)}개 트렌드 키워드 수집 완료")
            
        except Exception as e:
            print(f"⚠ 크롤링 오류: {e}")
            import traceback
            traceback.print_exc()
        
        finally:
            browser.close()
    
    # 결과 반환
    return {
        "total_trends": len(trends),
        "trends": trends
    }


In [ ]:
# Cell 4: 실행 및 결과 저장

print("▶ 구글 트렌드 실시간 인기 검색어 크롤링을 시작합니다...")
print(f"📦 최대 수집 개수: {MAX_TRENDS}개\n")

# 함수의 소스 코드를 가져와서 Python 파일로 저장
# 이렇게 하면 실제 Python 코드로 작성된 함수를 별도 프로세스에서 실행 가능
function_source = inspect.getsource(crawl_google_trends)

# 실행 스크립트 생성 (함수 + 실행 코드)
script_content = f"""from playwright.sync_api import sync_playwright
import json
import time
import re

# 함수 정의
{function_source}

# 설정값
EXCLUDED_TEXTS = {EXCLUDED_TEXTS!r}

# 함수 실행
if __name__ == "__main__":
    result = crawl_google_trends(
        headless={NOTEBOOK_HEADLESS!r},
        max_trends={MAX_TRENDS},
        excluded_texts=EXCLUDED_TEXTS
    )
    
    # JSON 파일로 저장
    output_json = "google_trends_results.json"
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    
    print(f"✅ JSON 파일 저장 완료: {{output_json}}")
    
    # 결과 미리보기
    print(f"\\n📋 수집된 트렌드 키워드 미리보기 (처음 15개):")
    for idx, trend in enumerate(result["trends"][:15], 1):
        print(f"  {{idx}}. {{trend['keyword']}}")
        if trend['link']:
            print(f"     링크: {{trend['link'][:70]}}...")
"""

# 임시 파일에 스크립트 저장
fd, script_path = tempfile.mkstemp(suffix="_crawl_google_trends.py", text=True)
os.close(fd)
with open(script_path, "w", encoding="utf-8") as f:
    f.write(script_content)

try:
    # 별도 프로세스에서 실행
    completed = subprocess.run(
        [sys.executable, script_path],
        capture_output=True,
        text=True,
        check=False
    )
    print(completed.stdout)
    if completed.returncode != 0:
        print("--- 프로세스가 에러로 종료되었습니다 (stderr) ---")
        print(completed.stderr)
    else:
        # JSON 파일 읽기
        output_json = "google_trends_results.json"
        if os.path.exists(output_json):
            with open(output_json, "r", encoding="utf-8") as f:
                result = json.load(f)
            print(f"\n✅ 크롤링 완료: 총 {result['total_trends']}개 키워드 수집")
finally:
    # 임시 파일 삭제
    try:
        os.remove(script_path)
    except Exception:
        pass


▶ 구글 트렌드 실시간 인기 검색어 크롤링을 시작합니다...
📦 최대 수집 개수: 80개

접속 중: https://trends.google.co.kr/trending?geo=KR
트렌드 콘텐츠 로드 대기 중...
스크롤하여 더 많은 트렌드 로드 중...

0. 페이지 네비게이션 테이블에서 키워드 추출 중...

1. .mZ3RIc 클래스에서 키워드 추출 중...

✅ 총 80개 트렌드 키워드 수집 완료
✅ JSON 파일 저장 완료: google_trends_results.json

📋 수집된 트렌드 키워드 미리보기 (처음 15개):
  1. psg 대 토트넘
     링크: https://trends.google.co.kr/trends/explore?q=%ED%86%A0%ED%8A%B8%EB%84%...
  2. 아스널 대 바이에른
     링크: https://trends.google.co.kr/trends/explore?q=%EC%95%84%EC%8A%A4%EB%82%...
  3. 리버풀 대 에인트호번
     링크: https://trends.google.co.kr/trends/explore?q=%EB%A6%AC%EB%B2%84%ED%92%...
  4. 홍콩 화재
     링크: https://trends.google.co.kr/trends/explore?q=%ED%99%8D%EC%BD%A9&date=n...
  5. 누리호 4차
     링크: https://trends.google.co.kr/trends/explore?q=%EB%82%98%EB%A1%9C%ED%98%...
  6. 날씨
     링크: https://trends.google.co.kr/trends/explore?q=%EB%82%B4%EC%9D%BC%20%EB%...
  7. 아틀레티코 대 인테르
     링크: https://trends.google.co.kr/trends/explore?q=%EC%9D%B8%ED%85%8C%EB%A5%...
  8. 두나무
     링크: htt